In [1]:
import openai

print(openai.__version__)

2.8.1


In [2]:

import os
import sys

os.chdir("..")
sys.path.append("src/")

In [3]:
from openai import OpenAI

model = 'gpt-5.1'
client = OpenAI()

user_prompt = 'Hi, my name is Maliheh'
response = client.responses.create(
    model=model,
    input=user_prompt
)

print(response.output_text)

Nice to meet you, Maliheh.  
What would you like to talk about or work on today?


In [4]:
from pydantic import BaseModel

class EventExtractInfo(BaseModel):
    name: str
    date: str
    participants: list[str]


response = client.responses.parse(
    model = model,
    input = [
        {'role': 'system', 'content': 'Extract the event information'},
        {'role': 'user', 'content': 'Alice and Bob are going to a science fair on Friday.'}
    ],
    text_format = EventExtractInfo
)

print(response.output_parsed)

name='science fair' date='Friday' participants=['Alice', 'Bob']


In [5]:
model = "gpt-4.1-mini"

response = client.responses.create(
    model = model,
    input = [
        {
            'role': 'user',
            'content': [
                {'type': 'input_text', 'text': "what's in this image?"},
                {'type': 'input_image', 'image_url': 'https://www.eatingwell.com/thmb/8MXOea6bVolkuLNQ1HhNt1tryIE=/750x0/filters:no_upscale():max_bytes(150000):strip_icc():format(webp)/article_7866255_foods-you-should-eat-every-week-to-lose-weight_-04-d58e9c481bce4a29b47295baade4072d.jpg'}
            ]
        }
    ]
)

print(response.output_text)

The image shows a variety of foods arranged on a light blue surface. The foods include:

- Purple cabbage wedge
- Brown rice in a small glass bowl
- Baby bok choy (three pieces)
- Raw salmon fillet on a white plate
- Broccolini stalks
- Whole and halved avocado
- Three whole eggs
- Sauerkraut in a small glass bowl
- Two apples
- Kimchi in a white bowl
- Two bars of dark chocolate
- Pistachios on a small white plate
- Chia seeds in a small glass bowl
- Wheat grains in a small bowl

These foods represent a mix of vegetables, fruits, protein sources, nuts, seeds, and fermented foods.


In [6]:
import base64

model = "gpt-4.1"

def encode_image_by_path(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


image_path = '/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp'
base64_image = encode_image_by_path(image_path)

response = client.responses.create(
    model=model,
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": "what's in this image?" },
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

print(response.output_text)

This image contains a variety of healthy foods, including:

- Bok choy (baby napa cabbage)
- Red cabbage
- Broccoli or broccolini
- Brown rice
- Farro (whole grain)
- Salmon fillet
- Eggs
- Sauerkraut (fermented cabbage)
- Kimchi (fermented vegetables, likely cabbage and chili)
- Apples
- Dark chocolate
- Avocado (whole and halved)
- Pistachio nuts
- Chia seeds

These foods are all commonly associated with a nutritious, balanced diet, and many are good sources of fiber, protein, healthy fats, and probiotics.


In [7]:
# lets you convert files (like images) into Base64 text
import base64

def encode_image_by_path(image_path):
    # Step 1: Open the image as bytes (binary data)
    # rb: read binary (images are not text, they're raw bytes)
    image_file = open(image_path, "rb")

    # Step 2: Read all the bytes from the image
    image_bytes = image_file.read()

    # Step 3: Convert those bytes into Base64 bytes
    base64_bytes = base64.b64encode(image_bytes)

    # Step 4: Convert Base64 bytes into a normal string
    base64_string = base64_bytes.decode("utf-8")

    # Step 5: Return the final Base64 string
    return base64_string


In [8]:
import base64

raw_data = b"\x89PNG\r\n\x1a\n"  # This is how real image bytes often look
print("RAW BYTES:", raw_data)

# Step 1: Base64 encode the raw bytes
base64_bytes = base64.b64encode(raw_data)
print("BASE64 (bytes):", base64_bytes)

# Step 2: Convert Base64 bytes → string
base64_string = base64_bytes.decode("utf-8")
print("BASE64 (string):", base64_string)


RAW BYTES: b'\x89PNG\r\n\x1a\n'
BASE64 (bytes): b'iVBORw0KGgo='
BASE64 (string): iVBORw0KGgo=


In [16]:
from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_path
from services.analysis.schemas import IngrediantsResponse
from services.prompts import MealScannerPrompts
    
model = 'gpt-4.1'
image_path = '/Users/malihehnorouzi/Desktop/foods-you-should-eat-every-week-to-lose-weight.webp'
base64_image = encode_image_by_path(image_path)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model, 
    system_prompt=MealScannerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=MealScannerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format = IngrediantsResponse
)

print(response)


name='Healthy Mixed Meal Ingredients' ingredients=[Ingredient(ingredient_name='Salmon fillet', portiont='1 fillet (about 4-6 oz)'), Ingredient(ingredient_name='Brown rice', portiont='1/2 cup uncooked'), Ingredient(ingredient_name='Farro (or another grain)', portiont='1/2 cup uncooked'), Ingredient(ingredient_name='Bok choy', portiont='2-3 stalks'), Ingredient(ingredient_name='Red cabbage', portiont='1/4 head'), Ingredient(ingredient_name='Broccolini', portiont='4-5 stalks'), Ingredient(ingredient_name='Avocado', portiont='1 whole'), Ingredient(ingredient_name='Eggs', portiont='3 whole'), Ingredient(ingredient_name='Apples', portiont='2 whole'), Ingredient(ingredient_name='Sauerkraut', portiont='1/2 cup'), Ingredient(ingredient_name='Kimchi', portiont='1/2 cup'), Ingredient(ingredient_name='Dark chocolate', portiont='6 squares'), Ingredient(ingredient_name='Pistachios', portiont='1/4 cup (shelled)'), Ingredient(ingredient_name='Chia seeds', portiont='2 tablespoons')]


In [17]:
from services.chat_gpt.gpt import ChatGPT
from services.analysis.schemas import IngrediantsResponse
from services.prompts import MealScannerPrompts


model = 'gpt-4.1'
image_url = 'https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg'

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_image_url(
    model=model, 
    system_prompt=MealScannerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=MealScannerPrompts.INGREDIENTS_USER_PROMPT,
    image_url=image_url,
    response_format = IngrediantsResponse
)

print(response)

name='Vietnamese Hu Tieu Nam Vang (Cambodian Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 bowl'), Ingredient(ingredient_name='Shrimp', portiont='2 pieces'), Ingredient(ingredient_name='Pork slices', portiont='3 pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Fresh herbs (mint, cilantro, green onions)', portiont='1 handful'), Ingredient(ingredient_name='Red chili peppers', portiont='1/2'), Ingredient(ingredient_name='Soup broth', portiont='1 cup')]


In [18]:
from pydantic import BaseModel

from services.chat_gpt.gpt import ChatGPT
from services.image_processing import encode_image_by_url
from services.analysis.schemas import IngrediantsResponse
from services.prompts import MealScannerPrompts

model = 'gpt-4.1'
image_url = 'https://images.pexels.com/photos/699953/pexels-photo-699953.jpeg'
base64_image = encode_image_by_url(image_url)

chat_gpt = ChatGPT()
response = chat_gpt.generate_image_response_by_base64_image(
    model=model, 
    system_prompt=MealScannerPrompts.INGREDIENTS_SYSTEM_PROMPT,
    user_prompt=MealScannerPrompts.INGREDIENTS_USER_PROMPT,
    base64_image=base64_image,
    response_format = IngrediantsResponse
)

print(response)





name='Vietnamese Hu Tieu Nam Vang (Phnom Penh Noodle Soup)' ingredients=[Ingredient(ingredient_name='Rice noodles', portiont='1 bowl'), Ingredient(ingredient_name='Shrimp', portiont='2 large pieces'), Ingredient(ingredient_name='Quail eggs', portiont='3 pieces'), Ingredient(ingredient_name='Pork (slices)', portiont='4-5 slices'), Ingredient(ingredient_name='Fresh herbs (cilantro, mint)', portiont='small handful'), Ingredient(ingredient_name='Green onions', portiont='1 tablespoon, chopped'), Ingredient(ingredient_name='Red chili peppers', portiont='1 teaspoon, sliced'), Ingredient(ingredient_name='Soup broth', portiont='Approximately 1 cup')]
